# Google AI Studio & Gemini API — Comprehensive Python Lab
### Agentic AI Course · Expanded educational notebook · October 2026

**Primary source:** [Google AI Studio — Get started](https://aistudio.google.com/docs/get-started) (user-provided link, tracking parameters removed).  
**Accessible official documentation:** [Google AI for Developers — Getting started](https://ai.google.dev/gemini-api/docs/get-started) (updated October 5, 2026).  
**Attribution:** Adapted from Google Gemini API documentation, with original explanations, exercises and classroom examples. Google documentation text is licensed under CC BY 4.0 except where noted; code samples under Apache 2.0. This notebook is *not* an official Google publication.

**Scope:** Python teaching adaptation of the guide, not a verbatim translation of its JavaScript/Java/Go/REST examples. Optional cells may require supported models, billable quotas, files or preview access. Do not run costly examples unintentionally.

**Learning goals:** authenticate, generate and stream responses, maintain state, process media, produce images and JSON, use built-in tools and local functions, understand managed agents, and submit background jobs.

## 1. Environment, API key and cost awareness
Get a key from [Google AI Studio](https://aistudio.google.com/apikey). Keep credentials in environment variables and never commit them to Git. API keys should not be exposed to frontend/browser apps. Billing and quotas depend on account and model. Each live cell can incur charges.

In [ ]:
%pip install -q -U google-genai pydantic pillow

In [ ]:
import os
from getpass import getpass
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key (hidden): ")
from google import genai
client = genai.Client()
MODEL = "gemini-3.8-flash"  # Replace with a model enabled for your account
print("Client configured. Selected model:", MODEL)

## 2. First call with the Interactions API
An **interaction** represents a turn and returns output plus metadata. The newer Interactions API uses `client.interactions.create(...)`; this is distinct from the older `client.models.generate_content(...)` API. Both may be encountered in examples online.

In [ ]:
interaction = client.interactions.create(model=MODEL, input="Explain agentic AI in 3 concise sentences.")
print(interaction.output_text)
print("Interaction ID:", interaction.id)
print("Status:", interaction.status)
print("Usage:", interaction.usage)

**Interpretation:** `output_text` provides a convenient textual result; `steps` may contain additional steps and metadata. Save only the information you are authorized to retain. Inspect structure safely.

In [ ]:
print("Step types:", [step.type for step in (interaction.steps or [])])

### Exercise — Prompt design
Ask for an agent objective, inputs, available tools, decision policy, and success metrics. Evaluate whether the model followed all five requirements.

In [ ]:
prompt = "Design a study-planning AI agent. Include: objective, inputs, tools, decision policy, success metrics."
print(client.interactions.create(model=MODEL, input=prompt).output_text)

## 3. Streaming
Streaming returns events as the response arrives rather than waiting for the complete result. Events can describe lifecycle changes and partial output; avoid assuming every event contains text.

In [ ]:
stream = client.interactions.create(model=MODEL, input="Explain the perceive–plan–act loop.", stream=True)
for event in stream:
    print(event)  # Explore event structure; in production filter text delta events.

## 4. Multi-turn conversations — stateful
A stateful conversation passes `previous_interaction_id`. The service retains context across turns according to API storage and retention policies. Do not confuse conversation state with durable application memory.

In [ ]:
first = client.interactions.create(model=MODEL, input="Our agent can read a catalog and create a shopping list.")
second = client.interactions.create(model=MODEL, input="What tools did I say it could use?", previous_interaction_id=first.id)
print(second.output_text)

## 5. Stateless conversation — explicit history
With `store=False`, the application manages complete history itself. The guide emphasizes preserving model-returned steps rather than reconstructing reasoning or function-call content manually. This is useful for data-flow control, but history grows with each turn.

In [ ]:
history = [{"type":"user_input", "content":[{"type":"text", "text":"My project is a campus helpdesk agent."}]}]
turn1 = client.interactions.create(model=MODEL, store=False, input=history)
history.extend(step.model_dump() for step in turn1.steps)
history.append({"type":"user_input", "content":[{"type":"text", "text":"What project am I working on?"}]})
turn2 = client.interactions.create(model=MODEL, store=False, input=history)
print(turn2.output_text)

## 6. Multimodal understanding
A multimodal request can combine text, images and audio. The example below requires a local `sample.jpg` and is intentionally guarded, so the notebook runs even without the file. Inspect sensitivity of student uploads and permissions before using external services.

In [ ]:
from pathlib import Path
import base64
p = Path("sample.jpg")
if p.exists():
    image64 = base64.b64encode(p.read_bytes()).decode()
    result = client.interactions.create(model=MODEL, input=[
        {"type":"text", "text":"Describe this image and list three visible objects."},
        {"type":"image", "data":image64, "mime_type":"image/jpeg"}
    ])
    print(result.output_text)
else:
    print("Optional exercise: add sample.jpg to the notebook folder, then rerun.")

### Image + remote audio exercise
Google's guide demonstrates mixing local image bytes and a remotely hosted audio URI. Remote media support and formats are model-dependent. Compare grounding, privacy implications and source reliability. [Official multimodal guidance](https://ai.google.dev/gemini-api/docs/get-started#5_multimodal_understanding).

## 7. Multimodal generation — image output
Some image models return base64-encoded image bytes. Image generation may involve distinct access permissions, rates and pricing. Below is an **optional** example: run only when a supported image model is available.

In [ ]:
RUN_IMAGE_GENERATION = False
if RUN_IMAGE_GENERATION:
    image_result = client.interactions.create(model="gemini-3.1-flash-image", input="Create a simple illustration of an AI assistant planning tasks.")
    from IPython.display import display, Image
    image_bytes = base64.b64decode(image_result.output_image.data)
    Path("generated_agent.png").write_bytes(image_bytes)
    display(Image(data=image_bytes))
else:
    print("Image generation skipped. Set RUN_IMAGE_GENERATION=True to opt in.")

## 8. Structured output with schemas
A schema provides machine-readable data for downstream automation. **Validation is still necessary**; format adherence is not proof that facts or recommendations are correct. Pydantic makes schema rules explicit.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
class AgentTask(BaseModel):
    title: str = Field(description="Brief task title")
    priority: int = Field(ge=1, le=5)
    owner: Optional[str] = None

# Structured response configuration follows the current Interactions API pattern.
structured = client.interactions.create(
    model=MODEL,
    input="List one task for a library-assistant agent.",
    response_format=AgentTask,
)
print(structured.output_text)

## 9. Built-in tools
Built-in tools allow the model to retrieve, calculate or execute operations with tool-specific constraints. Google's guide covers tools including web search and code execution. Verify tool availability and citations. The examples are opt-in to avoid surprising use or charges.

In [ ]:
RUN_BUILTIN_TOOLS = False
if RUN_BUILTIN_TOOLS:
    try:
        answer = client.interactions.create(model=MODEL, input="What are the newest developments in Gemini API? Cite sources.", tools=[{"type":"google_search"}])
        print(answer.output_text)
    except Exception as e:
        print("Tool/model unavailable or configuration needs adjusting:", type(e).__name__, str(e)[:250])
else:
    print("Built-in tool exercise skipped by default.")

## 10. Function calling — the critical agent building block
Function calling bridges model decisions and application code: (1) developer declares an interface, (2) model proposes a function name and arguments, (3) host application checks authorization and executes, (4) tool result is provided back to the model. **A model request alone does not authorize a real-world action.**

This offline simulation teaches the security boundary without relying on a changing wire schema.

In [ ]:
CATALOG = {"notebook": 4.5, "pen": 1.2}
def check_price(item: str) -> dict:
    key = item.lower().strip()
    if key not in CATALOG:
        return {"error": "unknown item"}
    return {"item":key, "price_usd":CATALOG[key]}

# Simulated model-produced tool call. In a real app, validate actual API-provided arguments.
proposed_call = {"name":"check_price", "arguments":{"item":"pen"}}
ALLOWED = {"check_price":check_price}
if proposed_call["name"] in ALLOWED:
    result = ALLOWED[proposed_call["name"]](**proposed_call["arguments"])
    print("Tool result:", result)

### Function-calling lab
Implement the live function-call protocol using the current guide. Inspect returned steps to locate function-call arguments, validate them, execute only allowlisted functions, and feed back the result. Be cautious: this interface may change across SDK versions. [Official reference](https://ai.google.dev/gemini-api/docs/get-started#9_call_your_own_functions).

## 11. Managed agents
The guide describes managed agents running in a remote sandbox, potentially with filesystem/code execution. Preview access, capabilities and billing can change. Keep this optional and do not supply secrets or sensitive documents without checking the runtime's policies.

In [ ]:
RUN_MANAGED_AGENT = False
if RUN_MANAGED_AGENT:
    managed = client.interactions.create(
        agent="antigravity-preview-09-2026",
        input="Write a Python script that computes and saves the first 20 Fibonacci numbers.",
        environment="remote"
    )
    print(managed.output_text)
else:
    print("Managed-agent preview exercise skipped.")

## 12. Background execution
Lengthy tasks may support submitting an interaction and checking its status later. This differs from a Jupyter kernel doing work indefinitely; the remote job must be polled and could fail or incur additional costs. Use finite retries and explicit opt-in.

In [ ]:
RUN_BACKGROUND = False
if RUN_BACKGROUND:
    import time
    job = client.interactions.create(model=MODEL, input="Draft a structured study plan for agentic AI.", background=True)
    for attempt in range(12):
        state = client.interactions.get(job.id)
        print("Status:", state.status)
        if state.status in ("completed", "failed", "cancelled"):
            print(getattr(state, "output_text", None))
            break
        time.sleep(5)
else:
    print("Background job skipped.")

## 13. Older `generate_content` API — migration awareness
Many tutorials use `client.models.generate_content()` and `client.chats.create()`. The earlier notebook followed this style. It remains useful for comparing concepts, but this revised lab **prioritizes the Interactions API** used by the current source guide.

In [ ]:
legacy = client.models.generate_content(model=MODEL, contents="Explain what a tool-using agent is in one sentence.")
print(legacy.text)

## 14. Reliability, governance, and responsible deployment
**Before production:** validate schemas, constrain tools, protect secrets, log access appropriately, consider prompt injection, disclose AI involvement where relevant, review costs/rate limits, and require human authorization for consequential actions. Distinguish model errors from transport, quota, policy, and configuration errors.

### Classroom capstone
Design a campus helpdesk agent that answers FAQs and queries a mock timetable. Provide: architecture, input schema, function contracts, permission model, failure modes, verification tests, and a lightweight risk register. **Do not** connect real school systems during the classroom exercise.

### Assessment questions
1. How does `previous_interaction_id` differ from explicit history?
2. Why must the host validate tool calls?
3. How is JSON schema compliance different from factual correctness?
4. What does streaming change in the user experience?
5. When should background or managed agent execution be avoided?

---
## References and mapping to original guide
**Original link supplied for this conversion:** https://aistudio.google.com/docs/get-started  
**Official accessible guide used for comparison:** https://ai.google.dev/gemini-api/docs/get-started  
**Google AI Studio API keys:** https://aistudio.google.com/apikey

**Source-guide topic mapping:** API key → §1; SDK/first call → §§1–2; streaming → §3; multi-turn → §§4–5; multimodal understanding → §6; multimodal generation → §7; structured output → §8; built-in tools → §9; function calling → §10; managed agent → §11; background execution → §12. Extra course-focused content → §§13–14.

**Editorial note:** This is an expanded *Python-only instructional adaptation* rather than a literal reproduction of Google's multi-language documentation. API names, model IDs, preview availability and billing may change; confirm against the linked official documentation before running examples. Live calls have not been tested here because no API credentials were provided.